In [1]:
%reset -f
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, gc, time, zipfile
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor
# 将上一级目录添加到搜索路径中
import sys
sys.path.append(os.path.abspath(".."))
from evaluator import evaluate_all  # 确保导入了你上传的评估文件

from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, accuracy_score, brier_score_loss, recall_score, precision_score)
from scipy import stats
import numpy as np

# 预测模型
from sklearn.linear_model import LogisticRegression    # Logit模型
from sklearn.ensemble import RandomForestClassifier     # 随机森林
from xgboost import XGBClassifier                       # XGBoost

# 评估与预处理
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler        # Logit 必须进行标准化
from sklearn.metrics import accuracy_score
from evaluator import evaluate_all

In [2]:
import joblib
file_path = "/root/autodl-tmp/DATA/data_bundles.pkl"
data_bundles = joblib.load(file_path)

In [3]:
# 函数
def undersample_indices(y, neg_pos_ratio=1, seed=42):
    """对训练集进行随机欠采样（Random Undersampling）"""

    # 确保标签为整数型 0/1
    y = np.asarray(y).astype(int)

    # 分别获取正样本和负样本的索引位置
    pos_idx = np.where(y == 1)[0]
    neg_idx = np.where(y == 0)[0]

    # 正样本数量
    n_pos = len(pos_idx)
    if n_pos == 0:
        raise ValueError("训练集中没有正样本，无法进行欠采样。")
    n_neg_need = min(len(neg_idx), n_pos * int(neg_pos_ratio))

    # 在负样本中随机无放回抽取所需数量
    rng = np.random.default_rng(seed)
    neg_sel = rng.choice(neg_idx, size=n_neg_need, replace=False)

    # 合并正样本与抽取的负样本索引
    sel = np.concatenate([pos_idx, neg_sel])
    # 打乱顺序，避免样本按类别排序
    rng.shuffle(sel)
    return sel

from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, accuracy_score, brier_score_loss)
from scipy import stats

# t 区间 CI（重复10次，df=9）
def t_ci(values, ci=0.95):
    v = np.asarray(values, dtype=float)
    v = v[~np.isnan(v)]
    n = len(v)
    if n < 2:
        return (np.nan, np.nan, np.nan)
    mean = v.mean()
    se = v.std(ddof=1) / np.sqrt(n)
    alpha = 1 - ci
    tcrit = stats.t.ppf(1 - alpha/2, df=n-1)
    lo = mean - tcrit * se
    hi = mean + tcrit * se
    return mean, lo, hi

def fmt_ci(mean, lo, hi, nd=4):
    if np.isnan(mean):
        return "NA"
    return f"{mean:.{nd}f}[{lo:.{nd}f}; {hi:.{nd}f}]"


metric_names = [
    "ROC-AUC", "F1", "Recall@1%", "Recall@5%"
]

def fit_predict_model(model, X_train, y_train, X_test, threshold=0.5):
    """
    统一封装：训练 + 预测概率 + 生成预测标签, 只需要替换 model 的构造即可
    """
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= threshold).astype(np.int8)

    # 某些模型（如logit）有 n_iter_，树模型没有
    n_iter = getattr(model, "n_iter_", None)
    if isinstance(n_iter, (list, np.ndarray)):
        n_iter = n_iter[0]
    return y_prob, y_pred, n_iter

In [6]:
from evaluator import evaluate_all
from joblib import Parallel, delayed
import joblib
import os
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import RandomForestClassifier

# RF 统一参数
RF_PARAMS_UNIFIED = {
    "n_estimators": 200,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": "sqrt",
    "bootstrap": True,
    "max_samples": 0.7,
    "class_weight": "balanced",  
}

RF_FIXED = dict(
    n_jobs=1,              # 并行 run 时必须为 1
)

# 参数
target = "insider_trading"
sets = ["set1", "set2", "set3"]
neg_pos_ratios = [1, 2, 5]
n_rep = 10
ci_level = 0.95
base_seed = 42
PARALLEL_JOBS = 20

# =========================
# 输出的指标列
# =========================
base_metrics = ["ROC-AUC", "F1"]
tail_metrics = ["Recall@0.10%","Recall@1.00%"]
metric_names = base_metrics + tail_metrics
# =========================
# ### [RF-MOD-2] 单次 RF run
# =========================
def one_run_rf(seed, ratio, X_train_full, y_train_full, X_test, y_test):

    sel = undersample_indices(y_train_full, neg_pos_ratio=ratio, seed=seed)
    X_train = X_train_full[sel]
    y_train = y_train_full[sel]

    model = RandomForestClassifier(
        **RF_PARAMS_UNIFIED,
        **RF_FIXED,
        random_state=seed,
    )

    model.fit(X_train, y_train)

    y_prob = model.predict_proba(X_test)[:, 1]

    metrics, _ = evaluate_all(
        y_true=y_test,
        y_prob=y_prob,
        base_thr=0.5,
        best_metric="f1",
        fig_path=None
    )

    return metrics, None

# 最终模型保存
def train_and_save_final(set_name, ratio,
                         X_train_full, y_train_full,
                         X_test, y_test):

    # === 新增：统一保存路径 ===
    base_save_dir = "/root/autodl-tmp/Table6"
    model_dir = os.path.join(base_save_dir, "models")
    pred_dir = os.path.join(base_save_dir, "preds")

    # 确保目录存在
    os.makedirs(model_dir, exist_ok=True)
    os.makedirs(pred_dir, exist_ok=True)

    sel = undersample_indices(y_train_full,
                              neg_pos_ratio=ratio,
                              seed=base_seed)

    X_train = X_train_full[sel]
    y_train = y_train_full[sel]

    model = RandomForestClassifier(
        **RF_PARAMS_UNIFIED,
        n_jobs=PARALLEL_JOBS,
        random_state=base_seed,
    )

    model.fit(X_train, y_train)

    model_path = os.path.join(model_dir,
                              f"rf_unified_{set_name}_ratio{ratio}.joblib")
    joblib.dump(model, model_path)

    y_prob = model.predict_proba(X_test)[:, 1]
    out = pd.DataFrame({
        "y_true": y_test.astype(int),
        "y_prob": y_prob.astype(float)
    })

    pred_path = os.path.join(pred_dir,
                             f"rf_pred_{set_name}_ratio{ratio}.csv")
    out.to_csv(pred_path, index=False)

    print(f"[FINAL] saved model: {model_path}")
    print(f"[FINAL] saved preds : {pred_path}")

# 主过程
tables = {}

for set_name in sets:
    print(f"\n==================== {set_name} ====================")

    train_df = data_bundles[set_name]["train"]
    test_df  = data_bundles[set_name]["test"]

    X_train_full = train_df[features].to_numpy(dtype=np.float32, copy=False)
    y_train_full = train_df[target].to_numpy(dtype=np.int8, copy=False)

    X_test = test_df[features].to_numpy(dtype=np.float32, copy=False)
    y_test = test_df[target].to_numpy(dtype=np.int8, copy=False)

    rows = []

    for ratio in neg_pos_ratios:
        print(f"\n--- 欠采样 负:正={ratio}:1 | 并行重复 {n_rep} 次 ---")
    
        store = {k: [] for k in metric_names}
        seeds = [base_seed + r for r in range(n_rep)]
    
        results = Parallel(
            n_jobs=PARALLEL_JOBS,
            backend="loky"
        )(
            delayed(one_run_rf)(
                seed, ratio,
                X_train_full, y_train_full,
                X_test, y_test
            )
            for seed in seeds
        )
    
        for met, n_iter in results:
            for k in metric_names:
                store[k].append(met.get(k, np.nan))
    
        row = {"Train undersample (neg:pos)": f"{ratio}:1"}
    
        for k in metric_names:
            row[k] = round(np.nanmean(store[k]), 4)
        rows.append(row)
        train_and_save_final(set_name, ratio,
                             X_train_full, y_train_full,
                             X_test, y_test)

    table = pd.DataFrame(rows)
    tables[set_name] = table
    display(table)


==================== set1 ====================

--- 欠采样 负:正=1:1 | 并行重复 10 次 ---
[FINAL] saved model: /root/autodl-tmp/Table6/models/rf_unified_set1_ratio1.joblib
[FINAL] saved preds : /root/autodl-tmp/Table6/preds/rf_pred_set1_ratio1.csv

--- 欠采样 负:正=2:1 | 并行重复 10 次 ---
[FINAL] saved model: /root/autodl-tmp/Table6/models/rf_unified_set1_ratio2.joblib
[FINAL] saved preds : /root/autodl-tmp/Table6/preds/rf_pred_set1_ratio2.csv

--- 欠采样 负:正=5:1 | 并行重复 10 次 ---
[FINAL] saved model: /root/autodl-tmp/Table6/models/rf_unified_set1_ratio5.joblib
[FINAL] saved preds : /root/autodl-tmp/Table6/preds/rf_pred_set1_ratio5.csv


,Train undersample (neg:pos),ROC-AUC,F1,Recall@0.10%,Recall@1.00%
0,1:1,0.7518,0.1486,0.478,0.478
1,2:1,0.7100,0.1531,0.478,0.478
2,5:1,0.7179,0.1449,0.478,0.478



==================== set2 ====================

--- 欠采样 负:正=1:1 | 并行重复 10 次 ---
[FINAL] saved model: /root/autodl-tmp/Table6/models/rf_unified_set2_ratio1.joblib
[FINAL] saved preds : /root/autodl-tmp/Table6/preds/rf_pred_set2_ratio1.csv

--- 欠采样 负:正=2:1 | 并行重复 10 次 ---
[FINAL] saved model: /root/autodl-tmp/Table6/models/rf_unified_set2_ratio2.joblib
[FINAL] saved preds : /root/autodl-tmp/Table6/preds/rf_pred_set2_ratio2.csv

--- 欠采样 负:正=5:1 | 并行重复 10 次 ---
[FINAL] saved model: /root/autodl-tmp/Table6/models/rf_unified_set2_ratio5.joblib
[FINAL] saved preds : /root/autodl-tmp/Table6/preds/rf_pred_set2_ratio5.csv


,Train undersample (neg:pos),ROC-AUC,F1,Recall@0.10%,Recall@1.00%
0,1:1,0.7423,0.1164,0.2951,0.4411
1,2:1,0.7389,0.1165,0.2989,0.4411
2,5:1,0.7423,0.1196,0.2669,0.4411



==================== set3 ====================

--- 欠采样 负:正=1:1 | 并行重复 10 次 ---
[FINAL] saved model: /root/autodl-tmp/Table6/models/rf_unified_set3_ratio1.joblib
[FINAL] saved preds : /root/autodl-tmp/Table6/preds/rf_pred_set3_ratio1.csv

--- 欠采样 负:正=2:1 | 并行重复 10 次 ---
[FINAL] saved model: /root/autodl-tmp/Table6/models/rf_unified_set3_ratio2.joblib
[FINAL] saved preds : /root/autodl-tmp/Table6/preds/rf_pred_set3_ratio2.csv

--- 欠采样 负:正=5:1 | 并行重复 10 次 ---
[FINAL] saved model: /root/autodl-tmp/Table6/models/rf_unified_set3_ratio5.joblib
[FINAL] saved preds : /root/autodl-tmp/Table6/preds/rf_pred_set3_ratio5.csv


,Train undersample (neg:pos),ROC-AUC,F1,Recall@0.10%,Recall@1.00%
0,1:1,0.8316,0.2741,0.3815,0.5653
1,2:1,0.8277,0.2792,0.3815,0.6001
2,5:1,0.7819,0.2809,0.3792,0.5924


In [7]:
SAVE_EXCEL = True
excel_path = "/root/autodl-tmp/Table6/rf_tables_with_tuning.xlsx"

# 自动创建路径中不存在的文件夹，防止路径错误
import os
os.makedirs(os.path.dirname(excel_path), exist_ok=True)

# 保存到Excel（每个set一个sheet）
if SAVE_EXCEL:
    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        for set_name, df_out in tables.items():
            df_out.to_excel(writer, sheet_name=set_name, index=False)
    print("\n已保存：", excel_path)


已保存： /root/autodl-tmp/Table6/rf_tables_with_tuning.xlsx
